# Fitting & Overfitting — Hands-on Tutorial

In this notebook we'll build intuition for:
1. **Choosing the right model** — fitting linear, quadratic, and sine models to the same data
2. **The overfitting trap** — what happens when you crank up model complexity
3. **Train/test split** — a practical tool to detect overfitting
4. **Overfitting detective** — diagnosing and fixing a broken model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from scipy.optimize import curve_fit

---
## Part 1: Choosing the Right Model

We'll generate noisy data from a known sine curve and try fitting three different models:
a line, a quadratic, and a sine function.

The goal: figure out which model is appropriate by looking at the **residuals**
(the difference between each data point and the model's prediction).

In [ ]:
# Generate noisy sine data
np.random.seed(7)
x = np.linspace(0, 2 * np.pi, 30)
y_true = np.sin(x)
y = y_true + np.random.normal(0, 0.2, len(x))

### Exercise: Write three model functions

Each function should take `x` (an array) and return the model's prediction.

**Hints:**
- Linear: `np.polyfit(x, y, 1)` returns coefficients, `np.polyval(coeffs, x)` evaluates
- Quadratic: same approach, but degree 2
- Sine: use `scipy.optimize.curve_fit`. It needs a function with signature `f(x, param1, param2, ...)`.
  A simple sine model is $y = A \sin(x + \phi)$. You'll need a reasonable initial guess —
  try `p0=[1.0, 0.0]` for $(A, \phi)$. `curve_fit` returns `(popt, pcov)` where `popt`
  is the array of best-fit parameter values.

In [ ]:
# Linear fit
# YOUR CODE HERE
# 1. Use np.polyfit to get the coefficients for a degree-1 polynomial
# 2. Use np.polyval to compute predictions
# Store the predictions in a variable called y_linear

raise NotImplementedError("Fit a linear model")

In [ ]:
# Quadratic fit
# YOUR CODE HERE
# Same idea as above, but with degree 2
# Store the predictions in y_quadratic

raise NotImplementedError("Fit a quadratic model")

In [ ]:
# Sine fit
# YOUR CODE HERE
#
# Step 1: Define the model function. curve_fit needs:
#   def sine_model(x, A, phi):
#       return A * np.sin(x + phi)
#
# Step 2: Call curve_fit:
#   popt, pcov = curve_fit(sine_model, x, y, p0=[1.0, 0.0])
#   p0 is the initial guess — curve_fit is iterative, so it needs
#   a starting point. Bad guesses can lead to bad fits.
#
# Step 3: Compute predictions using your fitted parameters:
#   y_sine = sine_model(x, *popt)
#   The * unpacks popt into the individual arguments A, phi
#
# Store the predictions in y_sine

raise NotImplementedError("Fit a sine model")

### Visualise the fits and their residuals

Run the cell below to see all three fits side by side.
The top row shows the model on the data, the bottom row shows the residuals.

In [ ]:
models = {
    'Linear': y_linear,
    'Quadratic': y_quadratic,
    'Sine': y_sine,
}
colors = ['#e74c3c', '#f39c12', '#27ae60']

fig, axes = plt.subplots(2, 3, figsize=(13, 6),
                         gridspec_kw={'height_ratios': [3, 1]})

for i, (name, y_pred) in enumerate(models.items()):
    residuals = y - y_pred

    # Top: data + fit
    axes[0, i].scatter(x, y, color='#2980b9', s=20, zorder=3)
    axes[0, i].plot(x, y_pred, color=colors[i], linewidth=2)
    axes[0, i].set_title(name, fontsize=12)
    if i == 0:
        axes[0, i].set_ylabel('y')

    # Bottom: residuals
    axes[1, i].scatter(x, residuals, color=colors[i], s=20)
    axes[1, i].axhline(0, color='grey', linewidth=0.5)
    axes[1, i].set_ylim(-1, 1)
    axes[1, i].set_xlabel('x')
    if i == 0:
        axes[1, i].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

### Think about it

- Which model's residuals look like random noise (no pattern)?
- Which model's residuals show clear structure? What does that tell you?
- The quadratic does better than the line — does that make it the right model?

---
## Part 2: The Overfitting Trap

What happens if we just keep adding parameters?

Below we generate sparse data from a simple linear trend. Use the slider to
increase the polynomial degree and watch the fit evolve.

In [ ]:
# Sparse data from a linear trend
np.random.seed(3)
x_sparse = np.linspace(0, 4, 8)
y_sparse = 0.5 * x_sparse + 1 + np.random.normal(0, 0.4, len(x_sparse))
x_dense = np.linspace(-0.5, 5.0, 200)


def plot_polyfit(degree):
    coeffs = np.polyfit(x_sparse, y_sparse, degree)
    y_fit = np.polyval(coeffs, x_dense)
    y_train_pred = np.polyval(coeffs, x_sparse)
    train_mse = np.mean((y_sparse - y_train_pred) ** 2)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(x_sparse, y_sparse, color='#2980b9', s=60, zorder=3, label='Data')
    ax.plot(x_dense, y_fit, color='#e74c3c', linewidth=2,
            label=f'Degree {degree} ({degree + 1} params)')
    ax.set_xlim(-0.5, 5.5)
    ax.set_ylim(-1, 6)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Training MSE = {train_mse:.4f}')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


interact(
    plot_polyfit,
    degree=IntSlider(value=1, min=1, max=7, step=1, description='Degree'),
);

### Think about it

- At what degree does the training MSE hit zero (or near-zero)?
- Does the fit at that degree look like it would predict new points well?
- Why does zero training error **not** mean a good model?

---
## Part 3: Train/Test Split

We saw that training error can be misleading. The fix: hold back some data,
fit only on the rest, and check how the model does on the held-out points.

### Exercise

1. Split the data into training and test sets
2. For polynomial degrees 1 through 7, fit on the training set and compute
   both training MSE and test MSE
3. Plot both errors vs. degree

In [ ]:
# A quadratic ground truth, so the best model is degree 2 (not the boundary).
np.random.seed(49)
x_all = np.linspace(0, 4, 20)
y_all = 0.4 * x_all**2 - 1.2 * x_all + 1.5 + np.random.normal(0, 0.4, len(x_all))

# Test points spread THROUGH the range (interpolation), not tacked on the end --
# so we measure generalisation, not extrapolation.
test_mask = np.zeros(len(x_all), dtype=bool)
test_mask[1::3] = True
x_train, y_train = x_all[~test_mask], y_all[~test_mask]
x_test,  y_test  = x_all[test_mask],  y_all[test_mask]

In [ ]:
degrees = range(1, 8)
train_errors = []
test_errors = []

for deg in degrees:
    # YOUR CODE HERE
    # 1. Fit a polynomial of this degree to the TRAINING data
    #    Hint: np.polyfit(x_train, y_train, deg)
    #
    # 2. Compute predictions on both train and test sets
    #    Hint: np.polyval(coeffs, x_train) and np.polyval(coeffs, x_test)
    #
    # 3. Compute MSE for each: np.mean((y_actual - y_predicted) ** 2)
    #
    # 4. Append the MSE values to train_errors and test_errors
    raise NotImplementedError("Compute train and test MSE for each degree")

In [ ]:
# Plotting boilerplate — plug in your train_errors and test_errors
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(list(degrees), train_errors, 'o-', color='#2980b9', linewidth=2,
        label='Training MSE')
ax.plot(list(degrees), test_errors, 's--', color='#e74c3c', linewidth=2,
        label='Test MSE')

ax.set_xlabel('Polynomial degree')
ax.set_ylabel('MSE')
ax.set_title('Training vs. Test Error')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Think about it

- Training error keeps going down as you add more parameters. Does test error?
- At which degree is test error lowest? That's the sweet spot.
- What happens to test error at high degrees? This is overfitting in action.

---
## Part 4: Overfitting Detective

You've been handed a pre-fit model and told it works great.
Your job: figure out if that's true, diagnose any problems, and fix them.

**Debugging tips:**
- Start by plotting the model on the data. Does it look reasonable?
- Plot the residuals. Do they look like noise, or is there a pattern?
- Check the model's behaviour *between* and *beyond* the data points.
  Overfitting often shows up as wild oscillations in gaps.
- Compare training error to test error — a big gap is the smoking gun.

In [ ]:
# The "production" model you've been given
np.random.seed(11)
x_prod = np.linspace(0, 5, 12)
y_prod = 0.3 * x_prod ** 2 - 0.5 * x_prod + 2 + np.random.normal(0, 0.5, len(x_prod))

# Someone fit a degree-11 polynomial (one less than the number of data points)
overfit_coeffs = np.polyfit(x_prod, y_prod, 11)

# Some held-out test data
np.random.seed(55)
x_test_prod = np.array([0.5, 1.3, 2.1, 3.0, 3.8, 4.5])
y_test_prod = 0.3 * x_test_prod ** 2 - 0.5 * x_test_prod + 2 + np.random.normal(0, 0.5, len(x_test_prod))

In [ ]:
# Step 1: Plot the model on the data
# YOUR CODE HERE
# - Scatter plot x_prod, y_prod (training data)
# - Evaluate the polynomial on a dense x grid (e.g., np.linspace(-0.5, 5.5, 200))
#   using np.polyval(overfit_coeffs, x_dense)
# - Plot the polynomial curve
# - Does anything look suspicious?

raise NotImplementedError("Plot the pre-fit model")

In [ ]:
# Step 2: Plot the residuals
# YOUR CODE HERE
# - Compute residuals: y_prod - np.polyval(overfit_coeffs, x_prod)
# - Scatter plot residuals vs x_prod
# - Draw a horizontal line at 0
# - Are the residuals suspiciously small? (near-zero = memorisation)

raise NotImplementedError("Plot residuals")

In [ ]:
# Step 3: Compare training vs test error
# YOUR CODE HERE
# - Compute MSE on training data (x_prod, y_prod)
# - Compute MSE on test data (x_test_prod, y_test_prod)
# - Print both. How do they compare?

raise NotImplementedError("Compare train vs test MSE")

In [ ]:
# Step 4: Fix it — fit a more appropriate model
# YOUR CODE HERE
# - Looking at the data, what kind of curve does it resemble?
# - Try a lower-degree polynomial
# - Compute and print the new train and test MSE
# - Plot your new fit alongside the original overfit curve

raise NotImplementedError("Fit a better model")

### Think about it

- How much did training MSE change between the degree-11 and your fix?
- How much did test MSE change?
- Why is a model that's slightly worse on training data actually *better* overall?